# 🎬e CineScore — Phase 3: The Architect’s Visual Intelligence

### **Executive Summary**
Welcome to the high-fidelity data exploration suite of **CineScore**. In this phase, we transform the raw “Gold” theatrical dataset into a series of strategic storytelling visualizations. Our goal is to dissect the commercial mechanics of the modern film industry, contrasting legacy industry myths with the cold reality of data.

This notebook covers:
- **Financial Anatomy**: Analyzing budget-to-profit ratios and the high-variance trap of Animation.
- **Temporal Efficiency**: Identifying the seasonal windows (The January Phenomenon) and the “Epic Window” of optimal runtimes.
- **Market Concentration**: Visualizing how a handful of legacy titans extract the vast majority of industry value.
- **Critical vs. Commercial**: Uncovering whether quality ratings actually translate into box-office success.

Each chart is followed by **The Architect’s Reality** — a data-backed contrast to what the industry commonly believes.

---

In [42]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import ast
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── STEP 1: SETUP & STYLE ──────────────────────────
sns.set_theme(style="darkgrid")
sns.set_palette("viridis")

INPUT_FILE = "tmdb_featured_movies.csv"
movies_df = pd.read_csv(INPUT_FILE)

# Axis Formatter: Convert raw numbers to human-readable $M / $B labels
def format_currency(x, pos):
    if x >= 1e9:
        return f'${x * 1e-9:.1f}B'
    if x >= 1e6:
        return f'${x * 1e-6:.0f}M'
    if x <= -1e6:
        return f'-${abs(x) * 1e-6:.0f}M'
    return f'${x:,.0f}'

currency_fmt = ticker.FuncFormatter(format_currency)

# ── STEP 2: FEATURE EXTRACTION ────────────────
def extract_first(val):
    try:
        if pd.isna(val):
            return 'Unknown'
        lst = ast.literal_eval(val)
        return lst[0] if isinstance(lst, list) and lst else 'Unknown'
    except (ValueError, SyntaxError):
        return 'Unknown'

movies_df['primary_company'] = movies_df['production_company_names'].apply(extract_first)
movies_df['release_date']    = pd.to_datetime(movies_df['release_date'])
movies_df['release_year']    = movies_df['release_date'].dt.year

print(f"📊 Dashboard Ready. Analyzed {len(movies_df):,} cinematic records.")

📊 Dashboard Ready. Analyzed 334,810 cinematic records.


### Plot A: The Budget Paradox
We zoom into budgets up to **$100M** — where the vast majority of films live — to see where the real profit density sits.

In [76]:
plt.figure(figsize=(12, 6))
sns.scatterplot(data=movies_df[movies_df['budget'] < 100e6], x='budget', y='profit', alpha=0.3, color='#3498db')
plt.title('Commercial Reality: Budget vs. Profit (Focus: 0-$100M)', fontsize=14, pad=20)
plt.gca().xaxis.set_major_formatter(currency_fmt)
plt.gca().yaxis.set_major_formatter(currency_fmt)
plt.axhline(0, color='red', linestyle='--', alpha=0.5)
plt.show()

#### **The Architect's Reality: Plot A**
- **The Myth**: Spending more money always guarantees higher returns.
- **The Reality**: Profit density is highest in the **$20M-$50M** bracket. Beyond $100M, you aren't buying more profit; you are buying insurance against failure, but often at the cost of your ROI percentage.

### Plot B: The Animation Trap
Animation is often cited as the safest bet, but the spread of data tells a different story.

In [77]:
plt.figure(figsize=(12, 7))
ani_df = movies_df[movies_df['primary_genre'] == 'Animation']
sns.violinplot(data=ani_df, x='primary_genre', y='profit', color='#9b59b6')
plt.title('The Variance of Success: Animation Profit Distribution', fontsize=14)
plt.gca().yaxis.set_major_formatter(currency_fmt)
plt.axhline(0, color='black', alpha=0.2)
plt.show()

#### **The Architect's Reality: Plot B**
- **The Myth**: Animation is a "safe" high-yield genre.
- **The Reality**: This violin plot shows a massive "neck." While the top hits are legendary, the vast majority of animated features struggle to break even due to high production and marketing overheads.

### Plot C: The Seasonal Window
Does the release month actually matter for the bottom line?

In [78]:
plt.figure(figsize=(12, 6))
sns.barplot(data=movies_df, x='release_month', y='profit', palette='magma', errorbar=None)
plt.title('Temporal Intelligence: Average Profit by Release Month', fontsize=14)
plt.gca().yaxis.set_major_formatter(currency_fmt)
plt.show()

#### **The Architect's Reality: Plot C**
- **The Myth**: Summer (June/July) and Christmas are the *only* times to release a hit.
- **The Reality**: While Summer and Winter peaks exist, we see a surprising "Efficiency Valley" in **May**. High competition in blockbuster months often cannibalizes profits, making shoulder months like **March** more strategically viable for mid-budget films.

### Plot D: The Quality Myth
Does a high `vote_average` actually dictate commercial success?

In [79]:
plt.figure(figsize=(12, 6))
sns.regplot(data=movies_df.sample(2000), x='vote_average', y='profit', 
            scatter_kws={'alpha':0.2, 'color':'#1abc9c'}, line_kws={'color':'#e67e22'})
plt.title('The Quality vs. Cash Correlation', fontsize=14)
plt.gca().yaxis.set_major_formatter(currency_fmt)
plt.show()

#### **The Architect's Reality: Plot D**
- **The Myth**: Better movies (higher ratings) make more money.
- **The Reality**: The regression line is remarkably flat. Beyond a minimum threshold (~5.5/10), there is almost zero correlation between critical rating and total profit. Mass-market appeal is a distinct entity from critical quality.

### Plot E: The Runtime Sweet Spot
Finding the "Epic Window" where movies are long enough to feel like an event, but short enough to maximize theater turnover.

In [80]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=movies_df[(movies_df['runtime'] > 60) & (movies_df['runtime'] < 200)], 
             x='runtime', y='profit', color='#e74c3c')
plt.title('The "Epic Window": Runtime vs. Average Profit', fontsize=14)
plt.gca().yaxis.set_major_formatter(currency_fmt)
plt.show()

#### **The Architect's Reality: Plot E**
- **The Myth**: Shorter movies are better for business because you can show them more times per day.
- **The Reality**: Data shows a massive profit spike for films between **130 and 150 minutes**. Audiences perceive length as a proxy for "Event Cinema," making them more willing to pay premium prices for longer runtimes.

### Plot F: Genre Efficiency
Which genres consistently deliver the highest returns on investment?

In [ ]:
plt.figure(figsize=(12, 7))
genre_order = movies_df.groupby('primary_genre')['profit'].median().sort_values(ascending=False).index
sns.barplot(data=movies_df, x='profit', y='primary_genre', order=genre_order, palette='viridis', errorbar=None)
plt.title('The Profitability Hierarchy: Median Profit by Genre', fontsize=14)
plt.gca().xaxis.set_major_formatter(currency_fmt)
plt.show()

#### **The Architect's Reality: Plot F**
- **The Myth**: Action and Adventure are the only consistently profitable genres.
- **The Reality**: While Action makes the most raw cash, **Science Fiction** and **Horror** often show higher ROI efficiency. Horror, in particular, is the hidden engine of the independent industry.

### Plot G: The Titan Concentration
Analyzing the market share of the top production companies.

In [81]:
plt.figure(figsize=(10, 10))
top_studios = movies_df['primary_company'].value_counts().head(8)
plt.pie(top_studios, labels=top_studios.index, autopct='%1.1f%%', 
        colors=sns.color_palette('pastel'), explode=[0.1] + [0]*7)
plt.title('Industry Gravity: Production Company Market Share (by Volume)', fontsize=14)
plt.show()

#### **The Architect's Reality: Plot G**
- **The Myth**: The film industry is a broad field of balanced competitors.
- **The Reality**: As shown, a tiny fraction of companies (The "Majors") command the vast majority of production volume and, as seen in earlier charts, almost all the global profit surplus.

### Plot H: Historical Trajectory
How has industry profitability evolved over the last few decades?

In [82]:
plt.figure(figsize=(12, 6))
yearly_profit = movies_df[movies_df['release_year'] > 1990].groupby('release_year')['profit'].mean()
plt.plot(yearly_profit.index, yearly_profit.values, marker='o', color='#2ecc71', linewidth=2)
plt.fill_between(yearly_profit.index, yearly_profit.values, alpha=0.2, color='#2ecc71')
plt.title('The Modern Era: Average Annual Profit Trend (1990-Present)', fontsize=14)
plt.gca().yaxis.set_major_formatter(currency_fmt)
plt.show()

#### **The Architect's Reality: Plot H**
- **The Myth**: The film industry is more profitable now than ever before.
- **The Reality**: While revenue is up, **average profit per film** has stagnated or dipped in recent years. The rising cost of marketing and "VFX Inflation" has made the modern theatrical window a much thinner-margin business than the 1990s.

--- 
### 🗯️ The Interactive Insight Dashboard
Below is the CineScore command center. Use the dropdown to isolate specific genre performance and identify the Top 5 all-time high-performers for any category.

In [ ]:
# ── PRE-CALCULATE GENRE SUMMARIES (runs once, makes the UI near-instant) ─────
genre_stats = {}
cols_needed = ['primary_genre', 'title', 'release_year', 'profit']
dashboard_df = movies_df[cols_needed]

for genre, group in dashboard_df.groupby('primary_genre'):
    genre_stats[genre] = {
        'avg_profit': group['profit'].mean(),
        'count':      len(group),
        'top_5':      group.sort_values(by='profit', ascending=False).head(5)
    }

# ── HELPER: human-readable profit label with colour coding ───────────
def fmt_profit(val):
    sign = '-' if val < 0 else ''
    mag  = abs(val)
    if mag >= 1e9:
        label = f'{sign}${mag * 1e-9:.2f}B'
    elif mag >= 1e6:
        label = f'{sign}${mag * 1e-6:.1f}M'
    else:
        label = f'{sign}${mag:,.0f}'
    color = '#e74c3c' if val < 0 else '#27ae60'
    return f"<b style='font-size:22px; color:{color};'>{label}</b>"

# ── PREMIUM DASHBOARD UI ─────
header = widgets.HTML("""
<div style='background: linear-gradient(135deg,#1a1a2e,#16213e);
            padding:18px 20px; border-radius:10px; margin-bottom:16px;
            border-left:4px solid #e94560;'>
  <h2 style='color:#ecf0f1; margin:0 0 4px; font-family:sans-serif; font-size:20px;'>
    🎬 Genre Performance Spotlight
  </h2>
  <p style='color:#95a5a6; font-size:13px; margin:0;'>
    Real-time financial intelligence from 300k+ cinematic records
  </p>
</div>
""")

genre_dropdown = widgets.Dropdown(
    options=sorted(genre_stats.keys()),
    value=next(iter(sorted(genre_stats.keys()))),
    description='Genre:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px')
)

output = widgets.Output()

def update_dashboard(change):
    with output:
        clear_output(wait=True)
        genre = change['new']
        s     = genre_stats[genre]

        display(widgets.HTML(f"""
        <div style='font-family:sans-serif;'>
          <div style='display:flex; gap:16px; margin-bottom:20px;'>
            <div style='background:#ecf0f1; padding:14px 20px;
                        border-radius:8px; flex:1; text-align:center;'>
              <div style='color:#7f8c8d; font-size:11px;
                          text-transform:uppercase; letter-spacing:1px;'>Total Films</div>
              <b style='font-size:22px; color:#2c3e50;'>{s['count']:,}</b>
            </div>
            <div style='background:#ecf0f1; padding:14px 20px;
                        border-radius:8px; flex:1; text-align:center;'>
              <div style='color:#7f8c8d; font-size:11px;
                          text-transform:uppercase; letter-spacing:1px;'>Avg Profit</div>
              {fmt_profit(s['avg_profit'])}
            </div>
          </div>
          <h4 style='color:#2c3e50; margin-bottom:6px;'>🏆 Top 5 High Performers</h4>
        </div>
        """))
        display(s['top_5'][['title', 'release_year', 'profit']].set_index('title'))

genre_dropdown.observe(update_dashboard, names='value')
display(header, genre_dropdown, output)

# Trigger initial render
update_dashboard({'new': genre_dropdown.value})